# Урок 06: Проектування баз даних та нормалізація
## Покроковий довідник

У цьому зошиті ми розглянемо базу даних IMDb і відповімо на питання: **Чому ця база даних розбита на окремі таблиці замість однієї гігантської?**

Ви дізнаєтесь:
1. **Первинні ключи (Primary Keys)** — як кожен рядок отримує унікальний ID
2. **Зовнішні ключи (Foreign Keys)** — як таблиці з'єднуються між собою
3. **Надлишковість (Redundancy)** — чому зберігання дублікатів це проблема

## Налаштування

In [ ]:
# Імпортуємо бібліотеки та підключаємося до бази даних
import pandas as pd
import sqlite3

conn = sqlite3.connect('imdb_class_2010plus.db')
print('Підключено до бази даних IMDb!')

## Частина 1: Первинні ключи (Primary Keys)

**Первинний ключ (primary key)** — це стовпець, який унікально ідентифікує кожен рядок у таблиці. Жодних двох рядків з однаковим значенням ключа.

Це як ваш студентський ID — жоден інший студент не має такого ж номера.

In [ ]:
# Доводимо що Title ID (tconst) унікальний у title_basics
query = '''
    SELECT 
        COUNT(*) as totalRows,
        COUNT(DISTINCT tconst) as uniqueIDs
    FROM title_basics
'''

df = pd.read_sql(query, conn)
total = df['totalRows'][0]
unique = df['uniqueIDs'][0]

print(f'Всього рядків: {total:,}')
print(f'Унікальних ID: {unique:,}')
print()
if total == unique:
    print('Всі рядки мають унікальний Title ID — tconst працює як первинний ключ!')

In [ ]:
# Чому НЕ можна використовувати назву фільму як ключ?
# Знаходимо назви що повторюються
query = '''
    SELECT primaryTitle, COUNT(*) as count
    FROM title_basics
    GROUP BY primaryTitle
    HAVING count > 1
    ORDER BY count DESC
    LIMIT 8
'''

df = pd.read_sql(query, conn)
print('Назви фільмів що повторюються:')
print(df.to_string(index=False))
print()
print('Декілька фільмів мають однакову назву!')
print('Тому ми використовуємо Title ID — він завжди унікальний.')

### Таблиця первинних ключів

| Таблиця | Первинний ключ | Приклад |
|---------|----------------|---------|
| title_basics | tconst (Title ID) | tt1375666 (Inception) |
| title_ratings | tconst (Title ID) | tt1375666 |
| name_basics | nconst (Person ID) | nm0000138 (Leonardo DiCaprio) |
| title_principals | tconst + ordering | tt1375666, 1 |

## Частина 2: Зовнішні ключи (Foreign Keys)

**Зовнішній ключ (foreign key)** — це стовпець в одній таблиці, який **посилається** на первинний ключ іншої таблиці. Це зв'язок між таблицями.

- `title_principals.tconst` → посилається на `title_basics.tconst`
- `title_principals.nconst` → посилається на `name_basics.nconst`

In [ ]:
# Сирі дані з таблиці акторів — тільки ID, не дуже зрозуміло
query = '''
    SELECT tconst, nconst, category
    FROM title_principals
    WHERE tconst = 'tt1375666'
    ORDER BY ordering
    LIMIT 5
'''

df = pd.read_sql(query, conn)
print('Сирі дані (тільки ID):')
print(df.to_string(index=False))

In [ ]:
# Тепер JOIN через зовнішні ключи — отримуємо справжні імена!
query = '''
    SELECT b.primaryTitle,
           n.primaryName,
           p.category
    FROM title_principals p
    JOIN title_basics b ON p.tconst = b.tconst
    JOIN name_basics n ON p.nconst = n.nconst
    WHERE p.tconst = 'tt1375666'
    ORDER BY p.ordering
    LIMIT 5
'''

df = pd.read_sql(query, conn)
print('Ті самі дані, з реальними іменами (через JOIN):')
print(df.to_string(index=False))
print()
print('Зовнішні ключі — це зв\'язки які роблять все це можливим!')

## Частина 3: Надлишковість — ворог бази даних

Уявіть одну гігантську таблицю де кожен рядок акторів також зберігає назву фільму, рік, рейтинг. Це **надлишковість** — зберігання одного й того ж факту багато разів.

In [ ]:
# Скільки надлишковості було б у поганому дизайні?
query = '''
    SELECT 
        COUNT(*) as totalCredits,
        COUNT(DISTINCT tconst) as uniqueTitles
    FROM title_principals
'''

df = pd.read_sql(query, conn)
credits = df['totalCredits'][0]
titles = df['uniqueTitles'][0]

print(f'Всього записів акторів: {credits:,}')
print(f'Унікальних фільмів: {titles:,}')
print()
print(f'ПОГАНИЙ дизайн: інформація про фільм зберігається {credits:,} разів')
print(f'ГАРНИЙ дизайн: інформація зберігається {titles:,} разів (по одному)')
print(f'Економія: {credits - titles:,} дублікатів усунуто!')

In [ ]:
# Проблема оновлень: надлишковість робить зміни небезпечними
query = '''
    SELECT b.primaryTitle, COUNT(*) as rowsToUpdate
    FROM title_principals p
    JOIN title_basics b ON p.tconst = b.tconst
    WHERE b.primaryTitle IN ('Inception', 'Interstellar', 'The Batman')
    GROUP BY b.primaryTitle
    ORDER BY rowsToUpdate DESC
'''

df = pd.read_sql(query, conn)
print('У ПОГАНОМУ дизайні, зміна назви фільму вимагає оновлення:')
print(df.to_string(index=False))
print()
print('У ГАРНОМУ дизайні: тільки 1 рядок у таблиці title_basics!')

## Підсумок: Три правила гарного дизайну бази даних

**1. Первинні ключи** — кожна таблиця має унікальний ідентифікатор
- Назви та імена НЕ достатньо унікальні для ключів. ID — краще.

**2. Зовнішні ключи** — таблиці з'єднуються через посилання на ID
- JOIN використовує зовнішні ключи для отримання даних з кількох таблиць

**3. Без надлишковості** — кожен факт зберігається рівно один раз
- "Inception" один раз у таблиці фільмів, не в кожному записі актора
- Оновлення безпечні, помилки друку не створюють неузгодженості

In [ ]:
# Закриваємо з'єднання
conn.close()
print('З\'єднання з базою даних закрито.')